In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr

import sys
sys.path.append('..')
from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

In [2]:
#data_folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/"
data_folder = '../Camera_Calibrations/Ximea_Camera/'
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

In [3]:
image_size = 20
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
R, G, B, wavelength = S_F.getpixelefficiency()
wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
camera_parameters = {}
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ['B', 'G', 'R']
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

In [4]:
n_photon_space = np.logspace(np.log10(500), np.log10(100000), 100)
n_bootstrap = 20000
background_photons = 40
pixel_size = 69
NA = 1.49

In [5]:
import types
smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma" :  1.5}
smoothing_function.extent =  1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [6]:
spectra = pl.read_csv('../Spectra/Nanoparticles/CL_spectra.csv')

dyes = spectra.columns[1:]

In [7]:
notch_filter = 'semrock-nf03-405-488-561-635e'
dichroic_mirror = 'semrock-di03-r405-488-561-635-t1-25x36'
nilered_filter = 'semrock-ff01-650-200-25'
shortpass_filter = 'semrock-bsp01-785r'
filters = []

In [8]:
save_folder = r'/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250729_CathodoLuminescence'
if not os.path.isdir(save_folder):
    os.makedirs(save_folder)

In [9]:
for dye in dyes:
    dyename = 'simulated_'+dye
    single_dye_spectrum = np.interp(wavelength, spectra['wv'], np.abs(spectra[dye]))
    single_dye_spectrum = single_dye_spectrum/np.sum(single_dye_spectrum)
    print("Analysing dye {}".format(dyename), end="\r",flush=True,)        
    MSF.test_fit_method(
                            dyename,
                            filters,
                            wavelength,
                            camera_parameters,
                            save_folder,
                            n_photon_space,
                            smoothing_function=smoothing_function,
                            starting_flag="CL_",
                            n_bootstrap=n_bootstrap,
                            background_photons=background_photons,
                            NA=NA,
                            pixel_size=pixel_size,
                            cpu_fraction=1,
                            single_dye_spectrum=single_dye_spectrum,
                            save_raw_results=True
                        )

LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 233.28task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 233.44task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 242.04task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 238.35task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 241.87task/s]


...

LM fitting: 100%|████████████████████████| 1800/1800 [00:08<00:00, 216.21task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:08<00:00, 217.07task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:08<00:00, 212.70task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:08<00:00, 212.09task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:08<00:00, 206.30task/s]


In [28]:
all_files = np.sort(os.listdir(save_folder))

In [40]:
Overall_XR = xr.DataArray(data=np.zeros([len(n_photon_space), n_bootstrap, len(dyes), 3]),
                          coords=[n_photon_space, np.arange(n_bootstrap)+1, dyes, ['B', 'G', 'R']],
                         dims=['Photons', 'Bootstrap_Repeat', 'Dye', 'Colour'])

In [41]:
for i, dye in enumerate(dyes):
    files = np.sort([os.path.join(save_folder, x) for x in all_files if dye in x and 'rawresults' in x and '#' not in x])
    for j, file in enumerate(files):
        data = pl.read_csv(file)
        for k, colour in enumerate(np.array(['A_B', 'A_G', 'A_R'])):
            Overall_XR[j, :, i, k] = data[colour].to_numpy()

In [43]:
Overall_XR.to_netcdf(os.path.join(save_folder, 'Simulated_CL_database.nc'))